# Self-Consistency -- Complete 65 -> 100 (Colab)

**Goal:** finish the 65-sample checkpoint in `self_consistency_results.json`
out to 100 samples. Uses the same `seed=42` -> identical 100 problems;
the script handles the remaining 35 problems (Wang et al., 2022 -- 5
paths per problem, majority vote, temperature=0.7).

## One-time setup
1. **API Key:** Secrets panel -> add `GROQ_API_KEY` -> enable 'Notebook access'
2. **Data and checkpoint:** create `MyDrive/NLP_SelfConsistency/` with:
   - `data/gsm8k_test.json`           <- test data
   - `results/self_consistency_results.json`  <- existing 65-sample checkpoint

## Run
**Runtime -> Run all** -- the script resumes from the checkpoint, picking up
at problem 66. After every problem it writes to Drive; safe to interrupt
and restart.

Estimated time: 35 problems x 5 paths x 1 API call x 8s ~= **25 min**
(excluding any rate-limit waits).


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE    = '/content/drive/MyDrive/NLP_SelfConsistency'
DRIVE_DATA    = os.path.join(DRIVE_BASE, 'data')
DRIVE_RESULTS = os.path.join(DRIVE_BASE, 'results')

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted.')
print('Data dir   :', DRIVE_DATA)
print('Results dir:', DRIVE_RESULTS)

In [ ]:
# 2. Install Groq SDK
!pip install groq -q
print('groq package ready.')

In [ ]:
# 3. Load API key
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY').strip()
if not GROQ_API_KEY:
    raise ValueError('GROQ_API_KEY not found. Add it from the Secrets panel.')
print('API key loaded.')

In [ ]:
# Configuration
N_SAMPLES   = 100   # total problem target
N_RUNS      = 5     # CoT paths per problem
TEMPERATURE = 0.7
MODEL       = 'llama-3.1-8b-instant'
SEED        = 42

CHECKPOINT_FILE = os.path.join(DRIVE_RESULTS, 'self_consistency_results.json')
DATA_FILE       = os.path.join(DRIVE_DATA,    'gsm8k_test.json')

print(f'Target       : {N_SAMPLES} problems')
print(f'Paths/problem: {N_RUNS}  (temp={TEMPERATURE})')
print(f'Checkpoint   : {CHECKPOINT_FILE}')
print(f'Data         : {DATA_FILE}')

In [ ]:
# 4. Data file check
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Please upload gsm8k_test.json to Drive/NLP_SelfConsistency/data/.'
    )
print('Data file found.')

# Checkpoint check
if os.path.exists(CHECKPOINT_FILE):
    import json as _json
    with open(CHECKPOINT_FILE, encoding='utf-8') as _f:
        _ckpt = _json.load(_f)
    print(f'Existing checkpoint: {_ckpt.get("n_samples", len(_ckpt.get("results", [])))} problems,'
          f' acc={_ckpt.get("accuracy")}%')
else:
    print('No checkpoint; will start from scratch.')

In [ ]:
# 5. LLM client (Groq)
import time
from groq import Groq

client = Groq(api_key=GROQ_API_KEY, timeout=120.0)

# Connectivity smoke test
try:
    _test = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Say: hello'}],
        max_tokens=5,
    )
    print('Connectivity test:', _test.choices[0].message.content)
except Exception as e:
    print(f'CONNECTIVITY TEST FAILED: {type(e).__name__}: {e}')
    raise


def chat(prompt, temperature=0.0, max_tokens=1024):
    for attempt in range(8):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            time.sleep(8)
            return response.choices[0].message.content.strip()
        except Exception as e:
            err_type = type(e).__name__
            print(f'\n  Error [{attempt+1}/8] ({err_type}): {str(e)[:150]}')
            wait = 60 * (attempt + 1) if 'RateLimit' in err_type or '429' in str(e) else 15
            if attempt < 7:
                print(f'  waiting {wait}s...')
                time.sleep(wait)
    raise RuntimeError('Failed after 8 attempts.')

print('LLM client ready.')

In [ ]:
# 6. Wei et al. (2022) Appendix G -- 8 standard CoT exemplars (inline)
FEW_SHOT_EXAMPLES = [
    {
        'question': 'There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?',
        'answer':   'There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6 trees planted. The answer is 6.',
    },
    {
        'question': 'If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?',
        'answer':   'There are originally 3 cars. Then 2 more cars arrive. So there are 3 + 2 = 5 cars now. The answer is 5.',
    },
    {
        'question': 'Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?',
        'answer':   'Originally, Leah had 32 chocolates and her sister had 42. So in total they had 32 + 42 = 74 chocolates. After eating 35, they had 74 - 35 = 39 chocolates. The answer is 39.',
    },
    {
        'question': 'Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?',
        'answer':   'Jason started with 20 lollipops. He then gave some to Denny and ended up with 12. So he gave 20 - 12 = 8 lollipops to Denny. The answer is 8.',
    },
    {
        'question': 'Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?',
        'answer':   'Shawn started with 5 toys. He got 2 toys from his mom and 2 toys from his dad. So he got 2 + 2 = 4 more toys. In total he now has 5 + 4 = 9 toys. The answer is 9.',
    },
    {
        'question': 'There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?',
        'answer':   'There were originally 9 computers. From Monday to Thursday is 4 days. So 4 * 5 = 20 new computers were added. In total, there are now 9 + 20 = 29 computers. The answer is 29.',
    },
    {
        'question': 'Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?',
        'answer':   'Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33. The answer is 33.',
    },
    {
        'question': 'Olivia has $23. She bought five bagels for $3 each. How much money does she have left?',
        'answer':   'Olivia started with $23. She bought 5 bagels for $3 each, so she spent 5 * 3 = $15 on bagels. She now has 23 - 15 = $8 left. The answer is 8.',
    },
]

FEW_SHOT_STR = '\n\n'.join(
    f"Q: {e['question']}\nA: {e['answer']}" for e in FEW_SHOT_EXAMPLES
)
print(f'Few-shot prompt ready ({len(FEW_SHOT_EXAMPLES)} exemplars).')

In [ ]:
# 7. Load data (seed=42 -> identical 100 problems to the other experiments)
import json
import random

with open(DATA_FILE, encoding='utf-8') as f:
    all_data = json.load(f)

random.seed(SEED)
test_data = random.sample(all_data, N_SAMPLES)

print(f'Total test set: {len(all_data)}')
print(f'This run      : {len(test_data)} problems (seed={SEED})')

In [ ]:
# 8. Helpers
import re
from collections import Counter


def extract_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    if m:
        return float(m.group(1).replace(',', ''))
    nums = re.findall(r'[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[-1]) if nums else None


def gold_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    return float(m.group(1).replace(',', '')) if m else None


def majority_vote(answers):
    valid = [a for a in answers if a is not None]
    if not valid:
        return None
    return Counter(valid).most_common(1)[0][0]


print('Helpers ready.')

In [ ]:
# 9. Self-Consistency -- resume from checkpoint, write to Drive after each problem

results        = []
done_questions = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        ckpt = json.load(f)
    results        = ckpt.get('results', [])
    done_questions = {r['question'] for r in results}
    print(f'Checkpoint found: {len(results)} problems already done.')
else:
    print('No checkpoint; starting from scratch.')

total = len(test_data)

for i, item in enumerate(test_data, 1):
    if item['question'] in done_questions:
        print(f'[{i}/{total}] skipping (already done)', end='\r', flush=True)
        continue

    prompt = f"{FEW_SHOT_STR}\n\nQ: {item['question']}\nA:"
    print(f'[{i}/{total}] sampling {N_RUNS} paths...', end='\r', flush=True)

    sampled_answers   = []
    sampled_responses = []
    for _run in range(N_RUNS):
        response = chat(prompt, temperature=TEMPERATURE)
        sampled_answers.append(extract_answer(response))
        sampled_responses.append(response)

    predicted = majority_vote(sampled_answers)
    gold      = gold_answer(item['answer'])

    results.append({
        'question':        item['question'],
        'gold':            gold,
        'predicted':       predicted,
        'correct':         predicted == gold,
        'sampled_answers': sampled_answers,
        'vote_counts':     {str(k): v for k, v in Counter(
                              a for a in sampled_answers if a is not None
                            ).items()},
        'responses':       sampled_responses,
    })

    correct_so_far = sum(r['correct'] for r in results)
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'strategy':  f'self_consistency_{N_RUNS}runs',
            'model':     MODEL,
            'n_samples': len(results),
            'n_runs':    N_RUNS,
            'accuracy':  round(correct_so_far / len(results) * 100, 1),
            'results':   results,
        }, f, indent=2, ensure_ascii=False)

correct = sum(r['correct'] for r in results)
n       = len(results)
print(f'\nSelf-Consistency finished.')
print(f'Result: {correct}/{n} correct -- Accuracy = {correct/n*100:.1f}%')
print(f'Checkpoint: {CHECKPOINT_FILE}')